# Update Gas ATB Capex with Halcyon Regression Results

Builds 3 candidate versions of `inputs/plant_characteristics/gas_ATB_2024_moderate.csv`, replacing `capcost` for **Gas-CC** and **Gas-CT** in years **2026-2032** with the forecasts produced by `CCGT_gas_capex.ipynb` and `CT_gas_capex.ipynb`. All other technologies (`Gas-CC_H_1x1`, `Gas-CC_H_2x1`, `Gas-CT_aero`) and all other years are left untouched.

Using the forecast CSVs already exported by the two source notebooks (`ccgt_regression_forecast.csv`, `ct_regression_forecast.csv`). Run those notebooks first if you want to refresh the forecasts.

- **Version 1** (`gas_CAPEX_update_v1_ccgt_low.csv`): Gas-CT = CT regression, Gas-CC = CCGT cluster 0 (low-cost tier) regression
- **Version 2** (`gas_CAPEX_update_v2_ccgt_high.csv`): Gas-CT = CT regression, Gas-CC = CCGT cluster 1 (high-cost tier) regression
- **Version 3** (`gas_CAPEX_update_v3_ccgt_all.csv`): Gas-CT = CT regression, Gas-CC = CCGT all-data (no clustering) regression

In [1]:
import pandas as pd

ATB_PATH = "../../inputs/plant_characteristics/gas_ATB_2024_moderate.csv"

ccgt_forecast = pd.read_csv("ccgt_regression_forecast.csv").set_index("Year")
ct_forecast = pd.read_csv("ct_regression_forecast.csv").set_index("Year")["Cost_$/kW"]
years = ccgt_forecast.index

ccgt_forecast

,Cluster_0_cost_$/kW,Cluster_1_cost_$/kW,AllData_cost_$/kW
Year,,,
2026,1242.6,2037.8,1312.2
2027,1299.2,2074.4,1505.0
2028,1355.8,2111.0,1697.9
2029,1412.5,2147.7,1890.7
2030,1469.1,2184.3,2083.5
2031,1525.7,2220.9,2276.3
2032,1582.4,2257.5,2469.2


## Build the 3 ATB versions

In [2]:
atb_base = pd.read_csv(ATB_PATH)

def make_version(ccgt_col, out_path):
    atb = atb_base.copy()

    is_cc_forecast_years = (atb["i"] == "Gas-CC") & (atb["t"].isin(years))
    atb.loc[is_cc_forecast_years, "capcost"] = atb.loc[is_cc_forecast_years, "t"].map(ccgt_forecast[ccgt_col]).values

    is_ct_forecast_years = (atb["i"] == "Gas-CT") & (atb["t"].isin(years))
    atb.loc[is_ct_forecast_years, "capcost"] = atb.loc[is_ct_forecast_years, "t"].map(ct_forecast).values

    atb.to_csv(out_path, index=False)
    return atb[(atb["i"].isin(["Gas-CC", "Gas-CT"])) & (atb["t"].isin(years))]

In [3]:
# Version 1: CT regression + CCGT cluster 0 (low-cost tier)
v1 = make_version(
    "Cluster_0_cost_$/kW",
    "../../inputs/plant_characteristics/gas_CAPEX_update_v1_ccgt_low.csv",
)
v1

,i,t,capcost,fom,vom,heatrate
16,Gas-CC,2026,1242.6,33.1,2.10,6.300
17,Gas-CC,2027,1299.2,32.8,2.09,6.285
18,Gas-CC,2028,1355.8,32.4,2.07,6.269
19,Gas-CC,2029,1412.5,32.1,2.06,6.253
20,Gas-CC,2030,1469.1,31.8,2.04,6.238
21,Gas-CC,2031,1525.7,31.4,2.02,6.222
22,Gas-CC,2032,1582.4,31.1,2.01,6.206
57,Gas-CT,2026,1233.8,25.6,6.94,9.717
58,Gas-CT,2027,1367.9,25.4,6.94,9.717
59,Gas-CT,2028,1502.0,25.3,6.94,9.717


In [4]:
# Version 2: CT regression + CCGT cluster 1 (high-cost tier)
v2 = make_version(
    "Cluster_1_cost_$/kW",
    "../../inputs/plant_characteristics/gas_CAPEX_update_v2_ccgt_high.csv",
)
v2

,i,t,capcost,fom,vom,heatrate
16,Gas-CC,2026,2037.8,33.1,2.10,6.300
17,Gas-CC,2027,2074.4,32.8,2.09,6.285
18,Gas-CC,2028,2111.0,32.4,2.07,6.269
19,Gas-CC,2029,2147.7,32.1,2.06,6.253
20,Gas-CC,2030,2184.3,31.8,2.04,6.238
21,Gas-CC,2031,2220.9,31.4,2.02,6.222
22,Gas-CC,2032,2257.5,31.1,2.01,6.206
57,Gas-CT,2026,1233.8,25.6,6.94,9.717
58,Gas-CT,2027,1367.9,25.4,6.94,9.717
59,Gas-CT,2028,1502.0,25.3,6.94,9.717


In [5]:
# Version 3: CT regression + CCGT all-data (no clustering)
v3 = make_version(
    "AllData_cost_$/kW",
    "../../inputs/plant_characteristics/gas_CAPEX_update_v3_ccgt_all.csv",
)
v3

,i,t,capcost,fom,vom,heatrate
16,Gas-CC,2026,1312.2,33.1,2.10,6.300
17,Gas-CC,2027,1505.0,32.8,2.09,6.285
18,Gas-CC,2028,1697.9,32.4,2.07,6.269
19,Gas-CC,2029,1890.7,32.1,2.06,6.253
20,Gas-CC,2030,2083.5,31.8,2.04,6.238
21,Gas-CC,2031,2276.3,31.4,2.02,6.222
22,Gas-CC,2032,2469.2,31.1,2.01,6.206
57,Gas-CT,2026,1233.8,25.6,6.94,9.717
58,Gas-CT,2027,1367.9,25.4,6.94,9.717
59,Gas-CT,2028,1502.0,25.3,6.94,9.717


## Sanity check: compare new capcost vs. the original ATB values

In [6]:
compare = atb_base[(atb_base["i"].isin(["Gas-CC", "Gas-CT"])) & (atb_base["t"].isin(years))][["i", "t", "capcost"]]
compare = compare.rename(columns={"capcost": "ATB_original"})
compare = compare.merge(v1[["i", "t", "capcost"]].rename(columns={"capcost": "v1_low"}), on=["i", "t"])
compare = compare.merge(v2[["i", "t", "capcost"]].rename(columns={"capcost": "v2_high"}), on=["i", "t"])
compare = compare.merge(v3[["i", "t", "capcost"]].rename(columns={"capcost": "v3_all"}), on=["i", "t"])
compare.sort_values(["i", "t"]).reset_index(drop=True)

,i,t,ATB_original,v1_low,v2_high,v3_all
0,Gas-CC,2026,1202.4,1242.6,2037.8,1312.2
1,Gas-CC,2027,1191.6,1299.2,2074.4,1505.0
2,Gas-CC,2028,1180.8,1355.8,2111.0,1697.9
3,Gas-CC,2029,1170.1,1412.5,2147.7,1890.7
4,Gas-CC,2030,1159.3,1469.1,2184.3,2083.5
5,Gas-CC,2031,1148.5,1525.7,2220.9,2276.3
6,Gas-CC,2032,1137.8,1582.4,2257.5,2469.2
7,Gas-CT,2026,1075.2,1233.8,1233.8,1233.8
8,Gas-CT,2027,1066.4,1367.9,1367.9,1367.9
9,Gas-CT,2028,1057.6,1502.0,1502.0,1502.0
